# Homework 5: Forecasting, Feedback Loops, and Working with AI

MSE 125 — Spring 2026

## Setup

In [1]:
import os, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (9, 4.5)
plt.rcParams['font.size'] = 12

DATA_DIR = 'data'
GH_BASE = "https://github.com/stanford-mse-125/book/releases/download/data-v1"

### Load the EPA AQI data (Sacramento County)

Daily Air Quality Index for every U.S. county, filtered to **Sacramento
County, California**, 2021–2024. Lec 16 modelled Los Angeles and Mono;
Sacramento is your data.

In [2]:
def fetch_aqi_year(year):
    local = f"{DATA_DIR}/epa-air-quality/daily_aqi_by_county_{year}.csv"
    if os.path.exists(local):
        return pd.read_csv(local)
    os.makedirs(os.path.dirname(local), exist_ok=True)
    url = f"{GH_BASE}/daily_aqi_by_county_{year}.csv"
    print(f"  downloading {url}")
    urllib.request.urlretrieve(url, local)
    return pd.read_csv(local)

aqi_frames = [fetch_aqi_year(y) for y in range(2021, 2025)]
aqi = pd.concat(aqi_frames, ignore_index=True)
sac = aqi[(aqi['State Name'] == 'California') & (aqi['county Name'] == 'Sacramento')].copy()
sac['Date'] = pd.to_datetime(sac['Date'])
sac = sac.sort_values('Date').reset_index(drop=True)
print(f"Sacramento: {len(sac)} daily AQI rows, "
      f"{sac['Date'].min().date()} to {sac['Date'].max().date()}")
print(f"AQI: mean {sac['AQI'].mean():.1f}, max {sac['AQI'].max()}, "
      f"days above 150 ('unhealthy'): {(sac['AQI'] > 150).sum()}")

Sacramento: 1461 daily AQI rows, 2021-01-01 to 2024-12-31
AQI: mean 58.6, max 381, days above 150 ('unhealthy'): 8

### Load the NBA team-game data

Player-game logs for the 2021–22, 2022–23, and 2023–24 NBA regular
seasons. Lec 17 used this dataset to demonstrate Simpson’s paradox on
rest days. For this homework you’ll work at the **team-game** level: one
row per team per game, with the team’s points, opponent, home/away flag,
and outcome (W/L).

In [3]:
def fetch_nba():
    local = f"{DATA_DIR}/nba/nba_load_management.csv"
    if os.path.exists(local):
        return pd.read_csv(local)
    os.makedirs(os.path.dirname(local), exist_ok=True)
    url = f"{GH_BASE}/nba_load_management.csv"
    print(f"  downloading {url}")
    urllib.request.urlretrieve(url, local)
    return pd.read_csv(local)

nba_raw = fetch_nba()
nba_raw['GAME_DATE'] = pd.to_datetime(nba_raw['GAME_DATE'])

# Aggregate player-rows to team-game rows
team_pts = nba_raw.groupby(
    ['GAME_ID', 'GAME_DATE', 'SEASON', 'TEAM_ABBREVIATION', 'OPPONENT', 'HOME']
).agg(PTS=('PTS', 'sum')).reset_index()
# Recover team W/L: any player row's WL flag is the team's outcome
team_wl = nba_raw.groupby(['GAME_ID', 'TEAM_ABBREVIATION'])['WL'].first().reset_index()
team_wl['WIN'] = (team_wl['WL'] == 'W').astype(int)
nba = team_pts.merge(team_wl[['GAME_ID', 'TEAM_ABBREVIATION', 'WIN']],
                     on=['GAME_ID', 'TEAM_ABBREVIATION'])
nba = nba.sort_values('GAME_DATE').reset_index(drop=True)
print(f"NBA team-games: {len(nba):,} rows, {nba['SEASON'].nunique()} seasons, "
      f"{nba['TEAM_ABBREVIATION'].nunique()} teams")
print(f"Home win rate: {nba[nba['HOME'] == 1]['WIN'].mean():.3f}")

NBA team-games: 7,380 rows, 3 seasons, 30 teams
Home win rate: 0.556

## How to use this notebook

Write your answers in the cells marked `# Your code here` or *Your
answer here* below each question. Add more cells if you need them
(Insert \> Code cell or Text cell). Run each code cell with Shift+Enter.
Run all cells top-to-bottom before submitting so your outputs are
included.

## Submission

Submit your completed notebook (`.ipynb`) to Gradescope by the due date.

## Problem 1: Diagnose and fix a forecast that breaks on advisory days

You analyze air-quality data for the **Sacramento Metropolitan Air
Quality Management District**. The director wants a one-day-ahead AQI
forecast so she can decide whether to issue an advisory for tomorrow.
The advisory threshold she cares about is AQI \> 100 (“unhealthy for
sensitive groups” or worse). A junior analyst has already proposed a
recipe: lag features in a linear regression, like Lec 16 used on Los
Angeles. Your job is to build that baseline on Sacramento, **find where
it fails**, **propose a fix**, and refit. Parts (a) and (b) recreate the
lecture’s toolkit on a fresh county and add the advisory-day uncertainty
check the lecture didn’t run; part (c) is where you go past anything the
lecture did.

Use the Sacramento AQI dataset loaded above.

**(a)** Build the baseline. Fit a linear regression on Sacramento AQI
using lag features of your choosing (yesterday, rolling means over a few
windows, day-of-year, etc. — at least three lag features). Train on
2021–2023, test on 2024. Report (i) test R², (ii) test MAE, (iii) the
MAE of the naive baseline that predicts tomorrow = today, and (iv) a 95%
bootstrap prediction interval built by resampling the training residuals
(1,000 replicates). Report the interval’s average width and its
**empirical coverage** — the fraction of 2024 test days whose actual AQI
falls inside the bootstrap interval (we say a 95% interval is
“calibrated” when empirical coverage is near 95%).

In 2–3 sentences, tell the director whether the model is improving on
the naive baseline overall, and what the prediction-interval width and
coverage tell her about the *typical* forecast.

In [4]:
# Your code here

*Your answer here.*

**(b)** Find where the baseline breaks. Run a **walk-forward
validation** across the 12 months of 2024: for each month, retrain on
every prior date and test on that month’s AQI; record the month’s MAE
and the maximum AQI observed that month. Plot the monthly MAE alongside
the monthly max AQI (paired panel or twin-axis chart — your choice).
Then report the prediction interval’s coverage **on the subset of test
days where actual AQI exceeded 100** — the days the director’s advisory
turns on. (To subset: `mask = te['AQI'] > 100`, then compute coverage on
`te[mask]`, `lower[mask]`, `upper[mask]`.)

In 2–3 sentences, identify the worst month, explain in concept-level
terms (Lec 16) *why* the model fails there, and tell the director
whether her interval is worth its 95% label on the days that decide an
advisory.

In [5]:
# Your code here

*Your answer here.*

**(c)** Propose a fix and refit. Looking at what (b) surfaced, propose
**one** change to the modelling pipeline that would make the forecast
more useful on advisory days. Two paths are open:

-   **Uncertainty quantification.** Stratify the bootstrap residual pool
    by regime — for example, resample residuals separately for days
    where the recent two weeks contained any AQI \> 100 vs. days where
    they did not — so the prediction interval widens on smoke-suspect
    days.
-   **Point forecast.** Add a feature (a longer-window rolling maximum,
    a binary “any day in the past two weeks had AQI \> 100”, an
    interaction term) or switch to a non-linear model.

Pick one, explain in one sentence what your change is trying to capture,
implement it on the same temporal split (train ≤ 2023, test 2024), and
re-report (i) fire-month MAE and (ii) advisory-day (AQI \> 100) PI
coverage.

In 2–3 sentences, tell the director whether your change helped —
*honestly*, even if it didn’t. **A “didn’t help” verdict earns full
credit** when the implementation is correct and the explanation of why
it didn’t help is grounded. Name one residual concern that your fix does
not address.

In [6]:
# Your code here

*Your answer here.*

## Problem 2: Working with AI — build, audit, decide

> **A note on this problem.** This problem is intentionally open-ended
> and asks you to do something the course has *not* explicitly drilled:
> use an AI assistant to write a piece of data analysis, then turn
> around and audit it the way [Lec 17]() taught you to audit any
> analysis. We do not tell you which test to run, which features to
> choose, or which checklist items will catch the most. The point is to
> live the audit loop, not to recover an answer key.
>
> Grading rewards the *quality of your audit and your defense of the
> recommendation* — not your model’s accuracy or your AI assistant’s
> polish.
>
> **Required deliverables.** A prompt log, a runnable analysis, a
> completed five-cluster audit table for *both* reports, a decision log,
> and a one-page memo to your friend.
>
> **Grading rubric (45 pts).** 10 / 20 / 15 across the three parts.
> Per-part rubrics in the solutions file name the alternatives TAs
> should accept.

Your friend Alex has been watching NBA games all season and wants to
start betting. Sportsbooks quote a typical moneyline as **−110** on both
sides, meaning you risk \$110 to win \$100 — so any betting strategy
must win **at least 52.4%** of bets to break even on average. Alex texts
you:

> *“You’re the stats person. Can you build me a model that predicts
> who’ll win, backtest it on last season, and tell me whether I should
> actually start betting? I’ll cover dinner.”*

You have one season of held-out data (2023–24) and access to whatever AI
assistant you usually use (ChatGPT, Claude, Codex, Claude Code, Gemini,
Cursor — pick one). You decide to draft the analysis with the AI and
then audit the result yourself before sending Alex anything.

**(a) Build the analysis with AI (10 pts).**

Use the AI assistant of your choice (ChatGPT, Claude, Gemini, Codex,
Cursor, Claude Code, …) to write the analysis Alex asked for. The
deliverable is a runnable pipeline that takes the team-game data,
produces a held-out estimate of the strategy’s return per dollar wagered
at −110 odds, and reports the number of bets it would have placed across
the season. *You* decide which features to use, which model family, how
to split the data, and what model-confidence threshold defines “place
the bet.”

Paste the code your assistant produced into the cell below and run it.
(If your assistant cannot execute Python, you are the executor — paste,
run, and feed the output back. That is itself part of the lesson from
Lec 17 on one-shot versus tool-using agents.)

Below the code, keep a **prompt log**: the prompts you sent, with a
sentence after each noting what the AI returned and how you reacted
(cleanly run, wrong-and-corrected, unexpected detail, etc.). Strong logs
do *one* of: (i) name the specific friction the AI introduced and the
fix you made, or (ii) explain what your first prompt *pre-empted* (an
assumption the AI would otherwise have made, a leakage trap you
front-loaded against). “The AI got it right first try” with no further
reflection earns less credit than either path.

In [7]:
# Your AI-assisted code here. Multiple cells are fine.

*Your prompt log here.* (3–6 prompts, one short sentence each on what
came back.)

**(b) Audit both reports (20 pts).**

Read the **planted report** reproduced at the bottom of this notebook
(“Planted reference report — *NBA Win-Prediction Backtest*”). It is a
separate analysis a different student’s AI assistant produced for the
same brief; you will compare your own analysis against it.

**Part (b1).** *Quantitative re-check.* Before the qualitative audit,
pick **one** numerical claim the planted report makes — its 5-fold CV
accuracy of 0.608, its 70.4% win rate at threshold 0.60, or its +34.5%
ROI — and **recompute** the corresponding number from the data yourself
using a **temporal** train/test split (train on 2021–22 and 2022–23,
test on 2023–24). Report your number alongside the planted report’s
claim, and in one sentence note whether they agree, differ, or
contradict. Then add one **additional** quantitative artifact the
planted report does not include: a 95% bootstrap confidence interval on
the ROI at whatever threshold you tested. Report the interval and note
whether it includes zero.

In [8]:
# Your code here

*Your answer here.*

**Part (b2).** *Five-cluster audit.* Now apply the **five-cluster
critical-evaluation checklist** from Lec 17 to *both* your own analysis
(part a) and the planted report. For each report, identify **at least
three** distinct issues that span **at least three different** checklist
clusters (data; model; signal; claim; incentives and dynamics).

Fill in the audit table below. Cite the *specific line of code, number,
or sentence* that is the evidence — vague flags (“the analysis seems
biased”) earn no credit.

| Report | Cluster | Specific issue | Evidence (line / number / quote) | Why it matters for Alex’s decision |
|------|-------|-----------|------------------------|-------------------------|
| Your own |  |  |  |  |
| Your own |  |  |  |  |
| Your own |  |  |  |  |
| Planted |  |  |  |  |
| Planted |  |  |  |  |
| Planted |  |  |  |  |

*Add rows if you find more.*

For each report, write a short paragraph (3–5 sentences) naming the
**single most consequential issue** for the bet/no-bet decision and
explaining why.

*Your answer here.*

**(c) The memo (15 pts).**

Write a one-page memo to Alex (markdown cell below, ≤ 400 words). It
must:

-   state your **bet / don’t bet** recommendation;
-   cite the **two or three** audit findings that drive your
    recommendation (one of which must be from the *incentives and
    dynamics* cluster);
-   acknowledge **one** thing about your own analysis you’d want to fix
    before letting Alex bet a dollar of real money;
-   end with a one-line statement of what you’d ask Alex to do if she
    decided to ignore your advice and bet anyway.

Fill out the decision log first so the memo can lean on it.

| Decision                                     | What you chose | Why |
|----------------------------------------------|----------------|-----|
| Model family                                 |                |     |
| Feature set                                  |                |     |
| Train/test split                             |                |     |
| Probability threshold for placing a bet      |                |     |
| Position-sizing rule (or “flat \$X per bet”) |                |     |
| Whether to bet at all                        |                |     |

*Memo here.*

## Problem 3: Design around a feedback loop

Lec 16 closed with a question we did not answer: *“You build a model
that predicts which students will fail a course, and the university uses
it to assign tutoring resources. Is this a Weapon of Math Destruction?”*
Your turn.

Stanford’s Vice Provost for Undergraduate Education is considering a
pilot: train a model on five years of transcripts to flag, in week 2 of
each quarter, students at risk of failing their current courses. Flagged
students get a personal email from a tutor offering free 1-on-1 help.

**(a)** Apply the **Weapon of Math Destruction** test from Lec 16: of
the three WMD properties (unmeasurable outcome; negative consequences
for individuals; self-fulfilling feedback loop), which apply to this
proposal as currently designed, and which do not? Two to three sentences
per property.

*Your answer here.*

**(b)** Identify the **proxy** the model optimizes and the **goal** the
Vice Provost actually cares about. Name one realistic way the proxy
could move in a direction that makes the goal *worse* — that is, one way
the system could be **gamed** or **distorted** in deployment, by
students, by tutors, or by faculty.

*Your answer here.*

**(c)** Propose **one** concrete design change that breaks the feedback
loop (e.g., changes who decides, who sees what, how often the model
retrains, what counts as the outcome), and **one** metric the Vice
Provost should monitor after deployment that would catch the failure
mode you described in (b) if it started to happen. Two to four sentences
total.

*Your answer here.*

## Problem 4: Reflection — how AI changed your workflow

In one paragraph (≤ 250 words), describe how using AI assistants has
changed your data-analysis workflow this quarter. Be concrete: name a
specific moment the AI helped you do something faster or better than you
could have alone, and a specific moment the AI gave you something that
looked right but was wrong (or incomplete) — and what course concept
helped you catch it.

This problem is graded for substance and specificity, not length. A
short, concrete paragraph beats a long, generic one.

*Your answer here.*

## Planted reference report — *NBA Win-Prediction Backtest*

> *The following is the AI-drafted report referenced in Problem 2(b). It
> was produced by a different student’s AI assistant in a single
> conversation and is reproduced here verbatim. Read it critically.*

> **NBA Win-Prediction Backtest**
>
> *Submitted by: Jordan Patel · Tool used: ChatGPT (GPT-5), single
> conversation · Date: May 2026*
>
> **Brief.** A friend asked: “Can you build a model that predicts NBA
> winners and tell me whether I should bet?” I used three seasons of NBA
> game logs from `data/nba/nba_load_management.csv` (2021–22, 2022–23,
> 2023–24) and built a logistic regression with rolling team-strength
> features. The model beats a coin flip at a high level of statistical
> significance. **At the threshold I recommend (model confidence \>
> 60%), the model’s backtested return on each \$1 wagered at standard
> −110 odds is +34%.** I think she should bet.
>
> **Data preparation.**
>
> ``` python
> import pandas as pd
> import numpy as np
> from sklearn.linear_model import LogisticRegression
> from sklearn.model_selection import KFold, cross_val_score
> from scipy import stats
>
> df = pd.read_csv('data/nba/nba_load_management.csv')
> df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
>
> # Aggregate player-rows to team-game rows
> team_pts = (df.groupby(['GAME_ID','GAME_DATE','SEASON',
>                         'TEAM_ABBREVIATION','OPPONENT','HOME'])
>               .agg(PTS=('PTS','sum')).reset_index())
> team_wl = df.groupby(['GAME_ID','TEAM_ABBREVIATION'])['WL'].first().reset_index()
> team_wl['WIN'] = (team_wl['WL'] == 'W').astype(int)
> nba = team_pts.merge(team_wl[['GAME_ID','TEAM_ABBREVIATION','WIN']],
>                      on=['GAME_ID','TEAM_ABBREVIATION'])
> nba = nba.sort_values(['TEAM_ABBREVIATION','GAME_DATE']).reset_index(drop=True)
>
> # Rolling team-strength features (each team's last 10 games)
> nba['team_pts_ma10'] = (nba.groupby('TEAM_ABBREVIATION')['PTS']
>     .transform(lambda s: s.shift(1).rolling(10, min_periods=3).mean()))
> nba['team_win_ma10'] = (nba.groupby('TEAM_ABBREVIATION')['WIN']
>     .transform(lambda s: s.shift(1).rolling(10, min_periods=3).mean()))
> nba['days_since']    = (nba.groupby('TEAM_ABBREVIATION')['GAME_DATE']
>     .diff().dt.days.fillna(7).clip(upper=10))
>
> opp = nba[['GAME_ID','TEAM_ABBREVIATION','team_pts_ma10','team_win_ma10']].rename(
>     columns={'TEAM_ABBREVIATION':'OPPONENT',
>              'team_pts_ma10':'opp_pts_ma10', 'team_win_ma10':'opp_win_ma10'})
> nba = nba.merge(opp, on=['GAME_ID','OPPONENT'])
> nba = nba.dropna(subset=['team_pts_ma10','team_win_ma10',
>                          'opp_pts_ma10','opp_win_ma10']).reset_index(drop=True)
> print(f"Final dataset: {len(nba):,} team-games")
> ```
>
> Output: `Final dataset: 7,284 team-games`
>
> **Model and evaluation.** I used 5-fold cross-validation on the full
> dataset to estimate held-out accuracy.
>
> ``` python
> feats = ['HOME','team_pts_ma10','team_win_ma10',
>          'opp_pts_ma10','opp_win_ma10','days_since']
>
> cv = KFold(n_splits=5, shuffle=True, random_state=7)
> scores = cross_val_score(LogisticRegression(max_iter=1000),
>                          nba[feats], nba['WIN'], cv=cv)
> print(f"5-fold CV accuracy: {scores.mean():.3f} ± {scores.std():.3f}")
> ```
>
> Output: `5-fold CV accuracy: 0.608 ± 0.012`
>
> The model is **highly statistically significant** vs. a coin flip:
>
> ``` python
> n = len(nba); wins = int(scores.mean() * n)
> p = stats.binomtest(k=wins, n=n, p=0.5, alternative='greater').pvalue
> print(f"n={n}, expected wins ~{wins}, binomial p vs. chance: {p:.2e}")
> ```
>
> Output: `n=7284, expected wins ~4429, binomial p vs. chance: 0.00e+00`
>
> **Backtest at −110 odds.** A −110 line means risking \$110 to win
> \$100, so any positive-edge strategy must clear a **52.4% win rate**.
> To select only the model’s most confident plays, I keep games where
> the model’s predicted win probability exceeds **0.60**.
>
> ``` python
> tr = nba[nba['SEASON'].isin(['2021-22','2022-23'])]
> te = nba[nba['SEASON'] == '2023-24'].copy()
> m = LogisticRegression(max_iter=1000).fit(tr[feats], tr['WIN'])
> te['prob'] = m.predict_proba(te[feats])[:, 1]
>
> bets = te[te['prob'] > 0.60]
> win_rate = bets['WIN'].mean()
> roi_per_dollar = (210 * win_rate - 110) / 110
>
> print(f"Bets placed: {len(bets):,}")
> print(f"Win rate:    {win_rate:.3f}")
> print(f"ROI per $1 wagered at -110: {roi_per_dollar:+.2%}")
> ```
>
> Output:
>
>     Bets placed: 565
>     Win rate:    0.704
>     ROI per $1 wagered at -110: +34.48%
>
> A 70.4% win rate on 565 bets is overwhelming evidence the model
> identifies winning teams. Across the 2023–24 season the strategy would
> have made roughly \$0.34 on every dollar wagered.
>
> **Recommendation.** Bet. The model is statistically significant (p ≈
> 0), accuracy is comfortably above coin flip, and the high-confidence
> bets clear the break-even rate by an enormous margin. A flat \$100 per
> qualifying bet would have generated about \$19,500 in expected profit
> across last season alone.
>
> *— end of planted report*